# Petroleum Supply Analysis — Phase 3

## 1. Verify Silver Data Availability
Verify access to the existing Silver table before building the
Gold dimensional model.

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

silver_table = "workspace.default.petroleum_silver"
execution_log_table = "workspace.default.petroleum_execution_logs"

if not spark.catalog.tableExists(silver_table):
    raise ValueError(f"Silver table does not exist: {silver_table}")

silver_df = spark.table(silver_table)

summary = silver_df.agg(
    F.count("*").alias("observation_count"),
    F.min("period").alias("earliest_month"),
    F.max("period").alias("latest_month"),
    F.countDistinct("series").alias("series_count"),
)

display(summary)
silver_df.printSchema()

%md
## 2. Profile Dimension Codes and Descriptions
Check missing dimension codes and identify codes associated with
multiple descriptions before building the Gold dimensions.

In [0]:
dimension_columns = [
    ("Product", "product", "product_name"),
    ("Area", "duoarea", "area_name"),
    ("Process", "process", "process_name"),
]

for label, code_column, name_column in dimension_columns:
    print(f"\n{label} dimension")

    # Summarize missing codes and descriptions.
    display(
        silver_df.agg(
            F.countDistinct(code_column).alias("distinct_codes"),
            F.sum(
                F.when(
                    F.col(code_column).isNull()
                    | (F.trim(F.col(code_column)) == ""),
                    1,
                ).otherwise(0)
            ).alias("missing_code_rows"),
            F.sum(
                F.when(
                    F.col(name_column).isNull()
                    | (F.trim(F.col(name_column)) == ""),
                    1,
                ).otherwise(0)
            ).alias("missing_description_rows"),
        )
    )

    # Identify codes with more than one non-null description.
    conflicting_names = (
        silver_df.groupBy(code_column)
        .agg(
            F.countDistinct(name_column).alias("description_count")
        )
        .filter(F.col("description_count") > 1)
    )

    display(conflicting_names)

# Inspect units separately to keep incompatible measures apart.
print("\nUnits used in Silver")
display(
    silver_df.groupBy("units")
    .count()
    .orderBy("units")
)

In [0]:
# Inspect the actual area codes and labels before modeling geography.
display(
    silver_df.select("duoarea", "area_name")
    .distinct()
    .orderBy("duoarea")
)

%md
## 3. Define the Gold Dimensional Model

### Fact Table Grain
Each fact row represents one monthly observation for one EIA series.
The logical business key is period + series.

### Planned Tables
| Table | Purpose | Primary Key |
|---|---|---|
| dim_date | Calendar date, month, quarter, and year | date_key |
| dim_product | Petroleum product code and description | product_key |
| dim_area | Original source area code and description | area_key |
| dim_process | Petroleum activity code and description | process_key |
| dim_unit | Measurement unit and description | unit_key |
| fact_petroleum_observation | Monthly observed value and dimension references | observation_key |

### Relationships
The fact table contains date_key, product_key, area_key,
process_key, and unit_key as foreign keys.
Each dimension has a one-to-many relationship with the fact table.

### Modeling Rules
- Preserve all original area codes, including codes sharing a label.
- Preserve series identifiers for observation-level traceability.
- Keep MBBL and MBBL/D measures separate.
- Do not sum ending stocks across months.
- Do not add national totals to regional figures.
- Do not add product totals to their subcategories.
- Preserve null values as missing observations, not zero.

In [0]:
# Configure Gold table names in the existing catalog and schema.
gold_table_prefix = "workspace.default.petroleum_gold"

gold_tables = {
    "date": f"{gold_table_prefix}_dim_date",
    "product": f"{gold_table_prefix}_dim_product",
    "area": f"{gold_table_prefix}_dim_area",
    "process": f"{gold_table_prefix}_dim_process",
    "unit": f"{gold_table_prefix}_dim_unit",
    "fact": f"{gold_table_prefix}_fact_observation",
}

for table_role, table_name in gold_tables.items():
    print(f"{table_role}: {table_name}")

%md
## 4. Define Explicit Gold Schemas
Define column names and data types for the five dimensions and
the observation fact table.

Dimension keys must be unique. Fact foreign keys must reference
existing dimension records. These rules will be validated during loading.

All Gold tables include load_timestamp.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType,
    TimestampType,
)

gold_schemas = {
    "date": StructType([
        StructField("date_key", IntegerType(), False),
        StructField("calendar_date", DateType(), False),
        StructField("year", IntegerType(), False),
        StructField("quarter", IntegerType(), False),
        StructField("month_number", IntegerType(), False),
        StructField("month_name", StringType(), False),
        StructField("year_month", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),

    "product": StructType([
        StructField("product_key", StringType(), False),
        StructField("product_code", StringType(), False),
        StructField("product_name", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),

    "area": StructType([
        StructField("area_key", StringType(), False),
        StructField("area_code", StringType(), False),
        StructField("area_name", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),

    "process": StructType([
        StructField("process_key", StringType(), False),
        StructField("process_code", StringType(), False),
        StructField("process_name", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),

    "unit": StructType([
        StructField("unit_key", StringType(), False),
        StructField("unit_code", StringType(), False),
        StructField("unit_description", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),

    "fact": StructType([
        StructField("observation_key", StringType(), False),
        StructField("date_key", IntegerType(), False),
        StructField("product_key", StringType(), False),
        StructField("area_key", StringType(), False),
        StructField("process_key", StringType(), False),
        StructField("unit_key", StringType(), False),
        StructField("series", StringType(), False),
        StructField("value", DoubleType(), True),
        StructField("record_hash", StringType(), False),
        StructField("silver_load_timestamp", TimestampType(), False),
        StructField("run_id", StringType(), False),
        StructField("load_timestamp", TimestampType(), False),
    ]),
}

for table_role, schema in gold_schemas.items():
    print(f"{table_role}: {len(schema.fields)} columns")

print("Gold schemas defined.")

%md
## 5. Create and Validate Gold Tables
Create the Gold Delta tables using the explicit schemas.
Preserve existing tables and verify their column names, types,
and nullability before loading data.

In [0]:
for table_role, table_name in gold_tables.items():
    expected_schema = gold_schemas[table_role]

    # Build the table definition from the explicit schema.
    column_definitions = ",\n".join(
        f"`{field.name}` {field.dataType.simpleString()}"
        + (" NOT NULL" if not field.nullable else "")
        for field in expected_schema.fields
    )

    # Existing tables and their data are preserved on reruns.
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {column_definitions}
        )
        USING DELTA
    """)

    actual_schema = spark.table(table_name).schema

    expected_fields = [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in expected_schema.fields
    ]

    actual_fields = [
        (field.name, field.dataType.simpleString(), field.nullable)
        for field in actual_schema.fields
    ]

    if actual_fields != expected_fields:
        raise ValueError(
            f"Schema mismatch for {table_name}.\n"
            f"Expected: {expected_fields}\n"
            f"Actual: {actual_fields}"
        )

    print(f"{table_role}: table ready — schema check PASS")

%md
## 6. Prepare Descriptive Dimensions
Build one dimension row per source code.
Reject missing codes, missing descriptions, and conflicting descriptions.

Generate stable keys from source codes so that keys remain unchanged
across repeated loads. This function prepares data without saving it.

In [0]:
dimension_specs = {
    "product": {
        "source_code": "product",
        "source_name": "product_name",
        "target_code": "product_code",
        "target_name": "product_name",
        "key_column": "product_key",
    },
    "area": {
        "source_code": "duoarea",
        "source_name": "area_name",
        "target_code": "area_code",
        "target_name": "area_name",
        "key_column": "area_key",
    },
    "process": {
        "source_code": "process",
        "source_name": "process_name",
        "target_code": "process_code",
        "target_name": "process_name",
        "key_column": "process_key",
    },
}


def prepare_descriptive_dimension(silver_source_df, *, dimension_role):
    """Prepare unique dimension members with stable source-code keys."""
    if dimension_role not in dimension_specs:
        raise ValueError(f"Unsupported dimension: {dimension_role}")

    spec = dimension_specs[dimension_role]
    code_column = spec["target_code"]
    name_column = spec["target_name"]

    members_df = silver_source_df.select(
        F.col(spec["source_code"]).alias(code_column),
        F.col(spec["source_name"]).alias(name_column),
    ).distinct()

    invalid_members = members_df.filter(
        F.col(code_column).isNull()
        | (F.trim(F.col(code_column)) == "")
        | F.col(name_column).isNull()
        | (F.trim(F.col(name_column)) == "")
    )

    if invalid_members.limit(1).count() > 0:
        raise ValueError(
            f"{dimension_role}: missing source codes or descriptions."
        )

    conflicting_codes = (
        members_df.groupBy(code_column)
        .count()
        .filter(F.col("count") > 1)
    )

    if conflicting_codes.limit(1).count() > 0:
        raise ValueError(
            f"{dimension_role}: a source code has conflicting descriptions."
        )

    # Hash only the source code and dimension role, not the description.
    members_df = members_df.withColumn(
        spec["key_column"],
        F.sha2(
            F.to_json(
                F.struct(
                    F.lit(dimension_role).alias("dimension"),
                    F.col(code_column).alias("code"),
                )
            ),
            256,
        ),
    )

    return members_df.select(
        spec["key_column"],
        code_column,
        name_column,
    )


print("Descriptive dimension preparation function ready.")

%md
## 7. Prepare the Date Dimension
Generate a continuous daily calendar covering complete years
from the earliest to the latest Silver observation.

Monthly observations reference the first day of their month.
Date keys use the YYYYMMDD integer format.

In [0]:
from datetime import date


def prepare_date_dimension(silver_source_df):
    """Build a continuous calendar covering the source observation years."""
    bounds = silver_source_df.agg(
        F.min("period").alias("minimum_period"),
        F.max("period").alias("maximum_period"),
    ).first()

    if bounds["minimum_period"] is None:
        raise ValueError("Cannot build a calendar from empty Silver data.")

    start_date = date(bounds["minimum_period"].year, 1, 1)
    end_date = date(bounds["maximum_period"].year, 12, 31)

    # Generate every calendar day, including dates without observations.
    calendar_df = spark.range(1).select(
        F.explode(
            F.sequence(
                F.lit(start_date),
                F.lit(end_date),
                F.expr("INTERVAL 1 DAY"),
            )
        ).alias("calendar_date")
    )

    return calendar_df.select(
        F.date_format("calendar_date", "yyyyMMdd")
        .cast("int")
        .alias("date_key"),
        F.col("calendar_date"),
        F.year("calendar_date").alias("year"),
        F.quarter("calendar_date").alias("quarter"),
        F.month("calendar_date").alias("month_number"),
        F.date_format("calendar_date", "MMMM").alias("month_name"),
        F.date_format("calendar_date", "yyyy-MM").alias("year_month"),
    )


print("Date dimension preparation function ready.")

%md
## 8. Prepare the Unit Dimension
Create one member per supported measurement unit.
Keep quantities and daily rates distinguishable for BI analysis.
Fail validation if an unsupported or missing unit is encountered.

In [0]:
def prepare_unit_dimension(silver_source_df):
    """Prepare supported measurement units with stable keys."""
    supported_units = ["MBBL", "MBBL/D"]

    units_df = silver_source_df.select(
        F.col("units").alias("unit_code")
    ).distinct()

    invalid_units = units_df.filter(
        F.col("unit_code").isNull()
        | ~F.col("unit_code").isin(supported_units)
    )

    if invalid_units.limit(1).count() > 0:
        raise ValueError(
            "Silver contains missing or unsupported measurement units."
        )

    return (
        units_df
        .withColumn(
            "unit_key",
            F.sha2(
                F.to_json(
                    F.struct(
                        F.lit("unit").alias("dimension"),
                        F.col("unit_code").alias("code"),
                    )
                ),
                256,
            ),
        )
        .withColumn(
            "unit_description",
            F.when(
                F.col("unit_code") == "MBBL",
                F.lit("Thousand barrels"),
            ).otherwise(
                F.lit("Thousand barrels per day")
            ),
        )
        .select(
            "unit_key",
            "unit_code",
            "unit_description",
        )
    )


print("Unit dimension preparation function ready.")

%md
## 9. Extend Execution Logging for Gold
Reuse the existing pipeline execution log table.
Add optional output and aggregation counts for Gold processing.
Preserve all existing Phase 2 audit records.

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

if not spark.catalog.tableExists(execution_log_table):
    raise ValueError(
        f"Execution log table does not exist: {execution_log_table}"
    )

# Add nullable metrics without replacing existing audit records.
existing_columns = set(spark.table(execution_log_table).columns)

for column_name in ["rows_output", "rows_aggregated"]:
    if column_name not in existing_columns:
        spark.sql(f"""
            ALTER TABLE {execution_log_table}
            ADD COLUMNS ({column_name} BIGINT)
        """)


def write_gold_execution_log(
    *,
    run_id,
    batch_id,
    load_type,
    source_reference,
    target_table,
    start_time,
    end_time,
    status,
    rows_read=None,
    rows_output=None,
    rows_aggregated=None,
    rows_inserted=None,
    rows_updated=None,
    error_message=None,
):
    """Append one Silver-to-Gold operation to the shared audit table."""
    if status not in {"Success", "Failure"}:
        raise ValueError("Status must be Success or Failure.")

    # Read the current schema, including the optional Gold metrics.
    log_schema = spark.table(execution_log_table).schema
    log_record = {field.name: None for field in log_schema.fields}

    log_record.update({
        "log_id": str(uuid4()),
        "run_id": run_id,
        "batch_id": batch_id,
        "load_type": load_type,
        "layer": "Silver-to-Gold",
        "source_reference": str(source_reference),
        "target_table": target_table,
        "start_time": start_time,
        "end_time": end_time,
        "status": status,
        "rows_read": rows_read,
        "rows_output": rows_output,
        "rows_aggregated": rows_aggregated,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
        "rows_quarantined": 0,
        "error_message": error_message,
        "load_timestamp": datetime.now(timezone.utc),
    })

    (
        spark.createDataFrame([log_record], schema=log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(execution_log_table)
    )


print("Gold execution logging ready.")

%md
## 10. Load Gold Dimensions with Audited MERGE
Insert new dimension members and update changed descriptions.
Preserve keys and timestamps for unchanged records.
Record success or failure for each dimension load.

In [0]:
def load_gold_dimension(
    silver_source_df,
    *,
    dimension_role,
    batch_id,
    load_type,
):
    """Prepare and merge one dimension with execution auditing."""
    if dimension_role not in {"date", "product", "area", "process", "unit"}:
        raise ValueError(f"Unsupported dimension: {dimension_role}")

    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    target_table = gold_tables[dimension_role]
    view_name = f"gold_dimension_{uuid4().hex}"

    rows_read = None
    rows_output = None
    rows_inserted = None
    rows_updated = None
    status = "Failure"
    error_message = None

    try:
        rows_read = silver_source_df.count()

        if rows_read == 0:
            raise ValueError("Cannot load a dimension from empty Silver data.")

        if dimension_role == "date":
            prepared_df = prepare_date_dimension(silver_source_df)
        elif dimension_role == "unit":
            prepared_df = prepare_unit_dimension(silver_source_df)
        else:
            prepared_df = prepare_descriptive_dimension(
                silver_source_df,
                dimension_role=dimension_role,
            )

        key_column = f"{dimension_role}_key"
        rows_output = prepared_df.count()

        # Prevent ambiguous MERGE matches or duplicate dimension keys.
        invalid_keys = (
            prepared_df.groupBy(key_column)
            .count()
            .filter(
                F.col(key_column).isNull()
                | (F.col("count") > 1)
            )
        )

        if invalid_keys.limit(1).count() > 0:
            raise ValueError("Dimension contains null or duplicate keys.")

        attribute_columns = [
            name for name in prepared_df.columns
            if name != key_column
        ]

        changed_condition = " OR ".join(
            f"NOT (target.`{name}` <=> source.`{name}`)"
            for name in attribute_columns
        )

        source_df = (
            prepared_df
            .withColumn(
                "load_timestamp",
                F.lit(start_time).cast("timestamp"),
            )
            .select(*gold_schemas[dimension_role].fieldNames())
        )

        source_df.createOrReplaceTempView(view_name)

        merge_result = spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {view_name} AS source
            ON target.`{key_column}` = source.`{key_column}`

            WHEN MATCHED AND ({changed_condition})
            THEN UPDATE SET *

            WHEN NOT MATCHED
            THEN INSERT *
        """).collect()

        if len(merge_result) != 1:
            raise RuntimeError("MERGE did not return operation metrics.")

        metrics = merge_result[0].asDict()

        if not {"num_inserted_rows", "num_updated_rows"}.issubset(metrics):
            raise RuntimeError("Required MERGE metrics are unavailable.")

        rows_inserted = int(metrics["num_inserted_rows"])
        rows_updated = int(metrics["num_updated_rows"])
        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_gold_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                source_reference=(
                    f"{silver_table}; dimension={dimension_role}; "
                    "scope=provided Silver DataFrame"
                ),
                target_table=target_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_output=rows_output,
                # No petroleum measure aggregation occurs in this operation.
                rows_aggregated=0,
                rows_inserted=rows_inserted,
                rows_updated=rows_updated,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "dimension": dimension_role,
        "run_id": run_id,
        "rows_output": rows_output,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
    }


print("Audited Gold dimension loader ready.")

%md
## 11. Load the Initial Gold Dimensions
Load all five dimensions from the current Silver table.
Use audited MERGE operations to preserve unchanged members on reruns.

This initializes dimensions from the complete current Silver snapshot.
It does not reload raw files or modify Bronze and Silver.

In [0]:
# Use the current persisted Silver data.
gold_source_df = spark.table(silver_table)

# Identify this initial Gold build in the audit logs.
gold_build_batch_id = f"gold_initial_{uuid4().hex}"

dimension_load_results = []

for dimension_role in ["date", "product", "area", "process", "unit"]:
    result = load_gold_dimension(
        gold_source_df,
        dimension_role=dimension_role,
        batch_id=gold_build_batch_id,
        load_type="full_load",
    )

    dimension_load_results.append(result)

    print(
        f"{dimension_role}: "
        f"prepared={result['rows_output']:,}, "
        f"inserted={result['rows_inserted']:,}, "
        f"updated={result['rows_updated']:,}"
    )

print("Initial Gold dimension loading complete.")

In [0]:
def prepare_gold_fact(silver_source_df):
    """Resolve dimension keys and prepare validated fact records."""
    source_count = silver_source_df.count()

    if source_count == 0:
        raise ValueError("Cannot prepare facts from empty Silver data.")

    invalid_keys = (
        silver_source_df.groupBy("period", "series")
        .count()
        .filter(
            F.col("period").isNull()
            | F.col("series").isNull()
            | (F.trim(F.col("series")) == "")
            | (F.col("count") > 1)
        )
    )

    if invalid_keys.limit(1).count() > 0:
        raise ValueError("Silver contains missing or duplicate business keys.")

    fact_df = silver_source_df.select(
        "period",
        "series",
        "product",
        "duoarea",
        "process",
        "units",
        "value",
        F.col("load_timestamp").alias("silver_load_timestamp"),
    )

    # Resolve the observation date against the calendar dimension.
    date_lookup = spark.table(gold_tables["date"]).select(
        F.col("calendar_date").alias("period"),
        "date_key",
    )

    fact_df = fact_df.join(date_lookup, on="period", how="left")

    dimension_joins = [
        ("product", "product", "product_code", "product_key"),
        ("area", "duoarea", "area_code", "area_key"),
        ("process", "process", "process_code", "process_key"),
        ("unit", "units", "unit_code", "unit_key"),
    ]

    for role, source_code, dimension_code, key_column in dimension_joins:
        lookup_df = spark.table(gold_tables[role]).select(
            F.col(dimension_code).alias(source_code),
            key_column,
        )

        fact_df = fact_df.join(
            lookup_df,
            on=source_code,
            how="left",
        )

    foreign_keys = [
        "date_key",
        "product_key",
        "area_key",
        "process_key",
        "unit_key",
    ]

    missing_match = F.lit(False)
    for key_column in foreign_keys:
        missing_match = missing_match | F.col(key_column).isNull()

    if fact_df.filter(missing_match).limit(1).count() > 0:
        raise ValueError("Some observations have no matching dimension member.")

    if fact_df.count() != source_count:
        raise ValueError("Dimension joins changed the observation count.")

    # The same period and series always produce the same observation key.
    fact_df = fact_df.withColumn(
        "observation_key",
        F.sha2(
            F.to_json(F.struct("period", "series")),
            256,
        ),
    )

    business_columns = foreign_keys + ["series", "value"]

    # Exclude processing timestamps so reruns do not trigger false updates.
    fact_df = fact_df.withColumn(
        "record_hash",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(name) for name in business_columns]),
                options={"ignoreNullFields": "false"},
            ),
            256,
        ),
    )

    return fact_df.select(
        "observation_key",
        *business_columns,
        "record_hash",
        "silver_load_timestamp",
    )


print("Gold fact preparation function ready.")

%md
## 13. Define the Audited Gold Fact Loader
Prepare dimension-linked observations and merge them into the Fact table.
Insert new observations and update only changed business content.
Log execution times, row counts, and failures.

In [0]:
def load_gold_fact(silver_source_df, *, batch_id, load_type):
    """Merge current Silver observations into Gold with execution logging."""
    run_id = str(uuid4())
    start_time = datetime.now(timezone.utc)
    target_table = gold_tables["fact"]
    view_name = f"gold_fact_{uuid4().hex}"

    rows_read = None
    rows_output = None
    rows_inserted = None
    rows_updated = None
    status = "Failure"
    error_message = None

    try:
        if load_type not in {"full_load", "incremental_load"}:
            raise ValueError("Unsupported load type.")

        if not isinstance(batch_id, str) or not batch_id.strip():
            raise ValueError("Batch ID must be a nonempty string.")

        rows_read = silver_source_df.count()

        # Validate observations and resolve their dimension keys.
        prepared_df = prepare_gold_fact(silver_source_df)
        rows_output = prepared_df.count()

        source_df = (
            prepared_df
            .withColumn("run_id", F.lit(run_id))
            .withColumn(
                "load_timestamp",
                F.lit(start_time).cast("timestamp"),
            )
            .select(*gold_schemas["fact"].fieldNames())
        )

        source_df.createOrReplaceTempView(view_name)

        merge_result = spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {view_name} AS source
            ON target.observation_key = source.observation_key

            WHEN MATCHED
                AND NOT (target.record_hash <=> source.record_hash)
            THEN UPDATE SET *

            WHEN NOT MATCHED
            THEN INSERT *
        """).collect()

        if len(merge_result) != 1:
            raise RuntimeError("MERGE did not return operation metrics.")

        metrics = merge_result[0].asDict()

        if not {"num_inserted_rows", "num_updated_rows"}.issubset(metrics):
            raise RuntimeError("Required MERGE metrics are unavailable.")

        rows_inserted = int(metrics["num_inserted_rows"])
        rows_updated = int(metrics["num_updated_rows"])
        status = "Success"

    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        raise

    finally:
        end_time = datetime.now(timezone.utc)

        try:
            write_gold_execution_log(
                run_id=run_id,
                batch_id=batch_id,
                load_type=load_type,
                source_reference=(
                    f"{silver_table}; "
                    "scope=provided current Silver observations"
                ),
                target_table=target_table,
                start_time=start_time,
                end_time=end_time,
                status=status,
                rows_read=rows_read,
                rows_output=rows_output,
                # This fact preserves observations without aggregation.
                rows_aggregated=0,
                rows_inserted=rows_inserted,
                rows_updated=rows_updated,
                error_message=error_message,
            )
        finally:
            spark.catalog.dropTempView(view_name)

    return {
        "run_id": run_id,
        "status": status,
        "rows_read": rows_read,
        "rows_output": rows_output,
        "rows_inserted": rows_inserted,
        "rows_updated": rows_updated,
    }


print("Audited Gold fact loader ready.")

%md
## 14. Load the Initial Gold Fact Table
Load the current Silver observations into the Gold Fact table.
Preserve the observation grain and record MERGE metrics in the audit log.

In [0]:
# Read current Silver observations for the initial Gold Fact load.
gold_source_df = spark.table(silver_table)

fact_load_result = load_gold_fact(
    gold_source_df,
    batch_id=gold_build_batch_id,
    load_type="full_load",
)

print(f"Status: {fact_load_result['status']}")
print(f"Silver rows read: {fact_load_result['rows_read']:,}")
print(f"Fact rows prepared: {fact_load_result['rows_output']:,}")
print(f"Fact rows inserted: {fact_load_result['rows_inserted']:,}")
print(f"Fact rows updated: {fact_load_result['rows_updated']:,}")

%md
## 15. Validate Gold Fact Completeness and Accuracy
Compare persisted Gold facts with the expected records from current Silver.
Verify row counts, observation-key uniqueness, dimension keys, and values.

In [0]:
current_silver_df = spark.table(silver_table)
actual_fact_df = spark.table(gold_tables["fact"])

# Rebuild the expected business records without writing to any table.
expected_fact_df = prepare_gold_fact(current_silver_df)

silver_count = current_silver_df.count()
fact_count = actual_fact_df.count()

if fact_count != silver_count:
    raise AssertionError(
        f"Row count mismatch: Silver={silver_count}, Gold={fact_count}"
    )

duplicate_keys = (
    actual_fact_df.groupBy("observation_key")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_keys.limit(1).count() > 0:
    raise AssertionError("Duplicate observation keys found in Gold.")

# Compare business content; processing metadata can differ by layer.
comparison_columns = [
    "observation_key",
    "date_key",
    "product_key",
    "area_key",
    "process_key",
    "unit_key",
    "series",
    "value",
    "record_hash",
]

expected_business_df = expected_fact_df.select(*comparison_columns)
actual_business_df = actual_fact_df.select(*comparison_columns)

missing_or_changed = expected_business_df.exceptAll(actual_business_df)
unexpected_records = actual_business_df.exceptAll(expected_business_df)

if missing_or_changed.limit(1).count() > 0:
    raise AssertionError("Gold contains missing or mismatched observations.")

if unexpected_records.limit(1).count() > 0:
    raise AssertionError("Gold contains unexpected observations.")

print(f"Silver observations: {silver_count:,}")
print(f"Gold observations: {fact_count:,}")
print("Observation key uniqueness: PASS")
print("Dimension keys and values match Silver: PASS")
print("Gold Fact reconciliation: PASS")

%md
## 16. Verify Gold Fact Idempotency
Reprocess unchanged Silver data.
Verify zero inserts and updates, and compare all Fact columns
against the saved pre-run Delta version.

In [0]:
fact_table = gold_tables["fact"]

# Capture the persisted table version before the rerun.
before_version = (
    spark.sql(f"DESCRIBE HISTORY {fact_table}")
    .orderBy(F.col("version").desc())
    .select("version")
    .first()["version"]
)

before_df = (
    spark.read
    .option("versionAsOf", str(before_version))
    .table(fact_table)
)

# Reprocess the same current Silver data.
fact_rerun_result = load_gold_fact(
    spark.table(silver_table),
    batch_id=f"gold_idempotency_{uuid4().hex}",
    load_type="full_load",
)

after_df = spark.table(fact_table).select(*before_df.columns)

if fact_rerun_result["rows_inserted"] != 0:
    raise AssertionError("Rerun unexpectedly inserted Fact rows.")

if fact_rerun_result["rows_updated"] != 0:
    raise AssertionError("Rerun unexpectedly updated Fact rows.")

# Compare every column, including run IDs and timestamps.
if before_df.exceptAll(after_df).limit(1).count() > 0:
    raise AssertionError("Existing Fact records changed or disappeared.")

if after_df.exceptAll(before_df).limit(1).count() > 0:
    raise AssertionError("Unexpected or modified Fact records appeared.")

print(f"Fact rows inserted: {fact_rerun_result['rows_inserted']}")
print(f"Fact rows updated: {fact_rerun_result['rows_updated']}")
print("Complete Fact records unchanged: PASS")
print("Gold Fact idempotency: PASS")

%md
## 17. Define the Gold Refresh Orchestrator
Refresh dimensions before loading the observation Fact.
Read a fixed Silver Delta version so all Gold operations use
the same source snapshot.

Use the existing audited MERGE loaders.
Stop execution if any dimension or Fact operation fails.

In [0]:
def refresh_gold(*, batch_id, load_type):
    """Refresh all Gold tables from one consistent Silver snapshot."""
    if not isinstance(batch_id, str) or not batch_id.strip():
        raise ValueError("Batch ID must be a nonempty string.")

    if load_type not in {"full_load", "incremental_load"}:
        raise ValueError("Unsupported load type.")

    # Pin one Silver version for consistent dimension and Fact processing.
    silver_version = (
        spark.sql(f"DESCRIBE HISTORY {silver_table}")
        .orderBy(F.col("version").desc())
        .select("version")
        .first()["version"]
    )

    source_df = (
        spark.read
        .option("versionAsOf", str(silver_version))
        .table(silver_table)
    )

    print(f"Refreshing Gold from Silver version: {silver_version}")

    dimension_results = []

    for dimension_role in ["date", "product", "area", "process", "unit"]:
        result = load_gold_dimension(
            source_df,
            dimension_role=dimension_role,
            batch_id=batch_id,
            load_type=load_type,
        )
        dimension_results.append(result)

        print(
            f"{dimension_role}: "
            f"inserted={result['rows_inserted']:,}, "
            f"updated={result['rows_updated']:,}"
        )

    # Run only after all dimensions have loaded successfully.
    fact_result = load_gold_fact(
        source_df,
        batch_id=batch_id,
        load_type=load_type,
    )
    agg_result = load_gold_aggregate(batch_id=batch_id, load_type=load_type)
    print(f"agg: inserted={agg_result['rows_inserted']:,}, updated={agg_result['rows_updated']:,}")
    print(
        f"fact: inserted={fact_result['rows_inserted']:,}, "
        f"updated={fact_result['rows_updated']:,}"
    )

    return {
        "silver_version": silver_version,
        "dimension_results": dimension_results,
        "fact_result": fact_result,
        "agg_result": agg_result
    }


print("Gold refresh orchestrator ready.")

%md
## 18. Verify Gold Refresh and Audit Coverage
Execute the Gold refresh orchestrator.
Verify that all six table operations have successful audit entries
with execution times and row metrics.

In [0]:
gold_refresh_batch_id = f"gold_refresh_check_{uuid4().hex}"

gold_refresh_result = refresh_gold(
    batch_id=gold_refresh_batch_id,
    load_type="full_load",
)

# Identify the exact operations from this refresh.
operation_results = (
    gold_refresh_result["dimension_results"]
    + [gold_refresh_result["fact_result"]]
)

refresh_run_ids = [
    result["run_id"] for result in operation_results
]

refresh_logs = (
    spark.table(execution_log_table)
    .filter(F.col("run_id").isin(refresh_run_ids))
    .filter(F.col("layer") == "Silver-to-Gold")
)

audit_rows = refresh_logs.select(
    "run_id",
    "target_table",
    "status",
    "start_time",
    "end_time",
    "rows_read",
    "rows_output",
    "rows_inserted",
    "rows_updated",
).collect()

if len(audit_rows) != 6:
    raise AssertionError("Expected exactly six Gold audit entries.")

if {row["run_id"] for row in audit_rows} != set(refresh_run_ids):
    raise AssertionError("Missing or duplicate operation logs.")

if {row["target_table"] for row in audit_rows} != set(gold_tables.values()):
    raise AssertionError("Audit entries do not cover all six Gold tables.")

for row in audit_rows:
    if row["status"] != "Success":
        raise AssertionError(f"Operation failed: {row['target_table']}")

    if (
        row["start_time"] is None
        or row["end_time"] is None
        or row["end_time"] < row["start_time"]
    ):
        raise AssertionError("Missing or invalid execution timestamps.")

    for metric in [
        "rows_read", "rows_output", "rows_inserted", "rows_updated"
    ]:
        if row[metric] is None or row[metric] < 0:
            raise AssertionError(f"Invalid audit metric: {metric}")

display(
    refresh_logs.select(
        "target_table",
        "status",
        "rows_read",
        "rows_output",
        "rows_inserted",
        "rows_updated",
        (
            F.col("end_time").cast("double")
            - F.col("start_time").cast("double")
        ).alias("duration_seconds"),
    ).orderBy("target_table")
)

print("All six Gold tables processed: PASS")
print("Gold audit coverage: PASS")

## 19. Optimize Gold Tables for Read Performance”

In [0]:
fact_table = gold_tables["fact"]

def table_files(table_name):
    d = spark.sql(f"DESCRIBE DETAIL {table_name}").first()
    return d["numFiles"], d["sizeInBytes"]

before_files, before_size = table_files(fact_table)

# Liquid Clustering on the most common BI filter/join keys.
spark.sql(f"ALTER TABLE {fact_table} CLUSTER BY (date_key, product_key, area_key)")
spark.sql(f"OPTIMIZE {fact_table}")

for role in ["date", "product", "area", "process", "unit"]:
    spark.sql(f"OPTIMIZE {gold_tables[role]}")

after_files, after_size = table_files(fact_table)

print(f"Fact files before: {before_files}, after: {after_files}")
print(f"Fact size before: {before_size:,} bytes, after: {after_size:,} bytes")
display(spark.sql(f"DESCRIBE DETAIL {fact_table}").select("clusteringColumns", "numFiles", "sizeInBytes"))